**Eye-Tracking & Neurocognitive Intelligence | Eye-Movement Modelling & Advanced AI | Presented by: Reza Saadatyar (2026)**

**01 | GazeBase Source, Target and Participant-Disjoint Main Cohort**

![Five-band source and split schematic from the completed ten-person pilot](../../images/01_data_and_experimental_protocol.svg)

**Next execution (not yet completed):** Default to **100 independent people**, with the option to change `MAX_PARTICIPANTS` in **Section 02 only** to any supported size, such as 300. The figure and earlier saved outputs depict the previous **ten-person exploratory pilot**, not the new main cohort.

**Source:** Official GazeBase v2.0 Round 1, Session 1, tasks FXS/HSS/RAN/TEX. Published Round 1 includes 322 people, but every selected person's four original files must pass acquisition checks. Data are downloaded, selectively extracted and processed entirely in **Colab** without Google Drive.

**Design:** Highest source IDs are reserved for main Test, with independent Train and Validation people; the window representation and train-only normalisation remain unchanged from the pilot. No Test predictions belong in this notebook. The final ZIP contains derived outputs only; the original 6.7-GB archive is deleted after successful selective extraction and may need a new download in another Runtime.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/Eye-Tracking-Neurocognitive-Intelligence


**01. Imports and Research Environment**


In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_movement_modelling")

# ------------------------ 2. Read-Only Source Adapter ------------------------
import sys
source_module = str(root / "eye_tracking_foundations" / "code" / "src")
if source_module not in sys.path:
    sys.path.insert(0, source_module)

# ----------------------- 3. Scientific Data Libraries ------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# -------------------------- 4. Modelling Utilities ---------------------------
from gaze_source import discover_records
from s2_data import (
    TASKS, CHANNELS, FEATURES, participant_split,
    build_source_tables, normalise_train_only, save_protocol,
)

# ----------------------------- 5. Output Folder ------------------------------
output_dir = root / "eye_movement_modelling" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100, figure_size=(9, 4))
print("Project output directory:", output_dir)
print("Shared original GazeBase source:", root / "data" / "raw" / "gazebase")


Local changes kept; continuing with the current checkout. Conflicting Git update deferred.
Project output directory: /content/Eye-Tracking-Neurocognitive-Intelligence/eye_movement_modelling/output
Shared original GazeBase source: /content/Eye-Tracking-Neurocognitive-Intelligence/data/raw/gazebase


**02. Configure, Acquire and Select Original GazeBase Recordings**

The following **single configuration cell** sets the number of participants, the random seed, Test/Validation fractions and the already approved temporal settings. Default: `MAX_PARTICIPANTS = 100`, source IDs 1001–1100; 300 participants would request 1001–1300. For this fixed Round 1/Session 1 protocol, the first N source IDs are a **deterministic convenience sample**, not a random population sample. Stop if any selected source is missing or invalid rather than silently reducing the cohort.

Source acquisition checks all **four original task recordings per selected person**, with no Google Drive connection. The complete official source ZIP is temporarily downloaded/verified only when originals are missing, and is deleted after extraction; original CSVs remain in the Colab checkout. Two small source-audit CSVs accompany the derived analysis outputs.

The proposed split allocates **20% to Test**, **20% of the remaining development people to Validation**, and the rest to Train (100 people: **64/16/20**; 300: **192/48/60**). Highest IDs form Test, kept independent of the previously examined pilot Test. Cohort size and the reserved Test membership must not be changed in response to main Test scores.


In [3]:
# ======================= 02. Main-Cohort Configuration =======================

# ---------------------------- 1. Research Cohort -----------------------------
MAX_PARTICIPANTS = 100         # Single edit for 100, 300, etc.
FIRST_SOURCE_ID = 1001        # Round 1 identifier sequence
SEED = 17
TEST_FRACTION = 0.20         # Of all selected people
VALIDATION_FRACTION = 0.20   # Of non-Test people

# ------------------------ 2. Temporal Input Protocol -------------------------
WINDOW_SAMPLES = 256
MAX_WINDOWS_PER_RECORD = 24

# --------------------------- 3. Validate Selection ---------------------------
if not 10 <= MAX_PARTICIPANTS <= 322:
    raise ValueError("Round 1 supports a maximum of 322 source people.")
if FIRST_SOURCE_ID != 1001:
    raise ValueError("This experiment uses Round 1 source IDs from 1001.")
if not (0 < TEST_FRACTION < 0.5 and 0 < VALIDATION_FRACTION < 0.5):
    raise ValueError("Split fractions must be between zero and 0.5.")
SOURCE_AUDIT_IDS = tuple(
    range(FIRST_SOURCE_ID, FIRST_SOURCE_ID + MAX_PARTICIPANTS)
)
print("Requested participants:", MAX_PARTICIPANTS)
print("Source IDs:", SOURCE_AUDIT_IDS[0], "-", SOURCE_AUDIT_IDS[-1])
print("Original files expected:", 4 * MAX_PARTICIPANTS)


Requested participants: 100
Source IDs: 1001 - 1100
Original files expected: 400


In [4]:
# ==================== 02. Acquire Original GazeBase CSVs =====================

# ------------------------ 1. Select Original Sources -------------------------
from s2_acquisition import ensure_gazebase_preflight

source_dir = root / "data" / "raw" / "gazebase"
source_audit = ensure_gazebase_preflight(
    raw_root=source_dir,
    download_root=Path("/content") / "gazebase_source_download",
    output_csv=output_dir / "01_source_acquisition_preflight.csv",
    requested_ids=SOURCE_AUDIT_IDS,
)

# -------------------------- 2. Audit Task Coverage ---------------------------
task_summary = source_audit.groupby("task", sort=True).agg(
    n_records=("record_id", "size"),
    n_people=("source_id", "nunique"),
    n_samples=("n_samples", "sum"),
    mean_valid_fraction=("valid_fraction", "mean"),
    source_size_bytes=("n_bytes", "sum"),
).reset_index()
task_summary.to_csv(
    output_dir / "01_source_acquisition_task_summary.csv", index=False,
)

# ---------------------------- 3. Inspect Sources -----------------------------
if source_audit.source_id.nunique() != MAX_PARTICIPANTS:
    raise ValueError("Source audit does not cover the requested people")
print("Source participants audited:", source_audit.source_id.nunique())
print("Source recordings audited:", len(source_audit))
print("Source directory:", source_dir)
display(task_summary)
display(source_audit.head(8))


Download source 1/4 (official Figshare): https://ndownloader.figshare.com/files/27039812
Downloaded 1.0 GB.
Downloaded 2.0 GB.
Downloaded 3.0 GB.
Downloaded 4.0 GB.
Downloaded 5.0 GB.
Downloaded 6.0 GB.
Verifying archive length and official published MD5...
Verified GazeBase archive ready for selective extraction.
Recovering 360 missing original CSVs from the archive.


FileNotFoundError: The verified archive does not contain all requested CSVs; still missing 24: S_1028_S1_FXS.csv, S_1028_S1_HSS.csv, S_1028_S1_RAN.csv, S_1028_S1_TEX.csv, S_1036_S1_FXS.csv, S_1036_S1_HSS.csv, S_1036_S1_RAN.csv, S_1036_S1_TEX.csv.

In [ ]:
# ================== 02. Select Original GazeBase Recordings ==================

# ------------------------ 1. User-Facing Cohort Size -------------------------
source_dir = root / "data" / "raw" / "gazebase"

# ----------------------- 2. Discover Complete Records ------------------------
inventory = discover_records(
    source_dir, round_id=1, session_id=1,
    tasks=TASKS, max_participants=MAX_PARTICIPANTS,
)
actual_people = inventory["participant_id"].nunique()
if actual_people != MAX_PARTICIPANTS:
    raise ValueError(
        f"Requested {MAX_PARTICIPANTS} complete participants but only "
        f"{actual_people} are available. Add authorised GazeBase source CSVs "
        "before increasing the cohort size."
    )
inventory.drop(columns=["path"]).to_csv(
    output_dir / "01_source_inventory.csv", index=False,
)
if set(inventory.record_id) != set(source_audit.record_id):
    raise ValueError("Source inventory differs from the audited cohort")
print("Complete participants:", actual_people)
print("Original task recordings:", len(inventory))
print("Source task labels:", sorted(inventory.task.unique()))
display(inventory[["record_id", "participant_id", "task", "relative_file"]].head(12))


**03. Register Participant-Disjoint Train, Validation and Test**

Assign whole participants, never individual recordings or windows, to disjoint roles. At the default 100-person cohort the intended counts are **64 Train, 16 Validation and 20 Test**. The new split and scientific parameters are recorded as current stage-01 outputs; the previous 6/2/2 pilot split must not be reused with them. Do not inspect main Test results until Notebook 05.


In [ ]:
# ================= 03. Assign Independent Participant Groups =================

# ------------------------- 1. Frozen Split Registry --------------------------
split = participant_split(
    inventory, seed=SEED, test_fraction=TEST_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
)
split.to_csv(output_dir / "01_split_assignments.csv", index=False)

# ------------------------ 2. Verify and Display Scope ------------------------
print("Study purpose:", split.study_purpose.iloc[0])
print("Independent people per partition:")
display(split.groupby("split").agg(
    n_people=("participant_id", "nunique"),
    n_task_records=("record_id", "count"),
).reset_index())
print("Participant sets:")
for role in ("train", "validation", "test"):
    people = sorted(split.loc[split.split.eq(role), "participant_id"].unique())
    print(role, ":", people)


**04. Build Record Features and Contiguous Motion Windows**

Stream each selected original recording, derive nine record-level gaze features and sample at most 24 non-overlapping 256-sample windows per recording (the settings are editable in Section 02). Do not bridge invalid samples or source timing gaps. The previous pilot had 40 records/960 windows; the next run's counts must be read from its new output, not inferred from that pilot.


In [ ]:
# ================== 04. Construct Gaze Features and Windows ==================

# ---------------------- 1. Bounded Sequence Parameters -----------------------
# --------------------- 2. Process Original Source Files ----------------------
record_features, windows, window_record_ids, window_labels = (
    build_source_tables(
        inventory, split, window_samples=WINDOW_SAMPLES,
        max_windows_per_record=MAX_WINDOWS_PER_RECORD,
    )
)
record_features.to_csv(output_dir / "01_record_features.csv", index=False)

# ------------------------- 3. Show Input Dimensions --------------------------
print("Source feature rows:", len(record_features))
print("Temporal windows / samples / channels:", windows.shape)
print("Raw temporal channels:", CHANNELS)
display(record_features[[
    "record_id", "participant_id", "task", "split",
    "valid_fraction", "n_windows", "speed_p95_dva_s",
]].head(12))


**05. Fit Temporal Normalisation on Training Participants Only**

Estimate four channel means/scales exclusively from Train windows and apply those values to Validation and reserved Test. Save the current normalised window archive and `01_sequence_normalisation.csv`. For large cohorts the compressed NPZ may exceed the repository's per-file size limit even though the Colab ZIP is still downloadable.


In [ ]:
# ==================== 05. Train-Only Motion Normalisation ====================

# ------------------------ 1. Calculate Train Moments -------------------------
scaled, centre, scale = normalise_train_only(
    windows, window_record_ids, split,
)
np.savez_compressed(
    output_dir / "01_sequence_windows.npz",
    x=scaled, y=window_labels, record_id=window_record_ids,
)
pd.DataFrame({
    "channel": CHANNELS, "train_mean": centre, "train_scale": scale,
}).to_csv(output_dir / "01_sequence_normalisation.csv", index=False)

# ------------------------- 2. Show Train Statistics --------------------------
print("Normalised windows:", scaled.shape)
display(pd.DataFrame({
    "channel": CHANNELS, "train_mean": centre, "train_scale": scale,
}))
print("No Validation or Test examples were fitted during normalisation.")


**06. Save the Protocol Table and Source Overview**

Record source ID range, selected count, seed, split fractions, temporal settings, data boundary and study purpose in `01_protocol.csv`. Generate the distribution plot from the new cohort's actual role assignments; the introductory five-band schematic continues to describe the prior pilot.


In [ ]:
# =================== 06. Save Protocol and Source Overview ===================

# -------------------- 1. Record Explicit Scientific Setup --------------------
protocol = save_protocol(
    output_dir / "01_protocol.csv",
    participants=MAX_PARTICIPANTS, seed=SEED,
    window_samples=WINDOW_SAMPLES,
    max_windows_per_record=MAX_WINDOWS_PER_RECORD,
    purpose=split.study_purpose.iloc[0],
    test_fraction=TEST_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    source_first_id=FIRST_SOURCE_ID,
)
display(pd.Series(protocol, name="Setting").to_frame())

# ---------------------- 2. Visualise Independent Units -----------------------
counts = split.groupby(["split", "task"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4.5))
counts.reindex(["train", "validation", "test"]).plot(
    kind="bar", ax=ax, rot=0,
)
ax.set(xlabel="Disjoint participant partition",
       ylabel="Original task recordings",
       title="GazeBase Task Availability by Participant Split")
ax.legend(title="Task")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "01_source_distribution.png")
plt.show()


**07. Check Current Stage Outputs and Export One ZIP**

Check the nine named stage-01 outputs, report any individual file larger than 50,000,000 bytes, and create `/content/eye_movement_modelling_Notebook_01_Outputs.zip`. The ZIP contains derived analysis inputs only, not the original source archive. Download it before ending Colab. In fresh Runtimes, the stage-specific importer in Notebooks 02–05 explicitly prioritises an uploaded current stage ZIP over older results from a GitHub checkout.


In [ ]:
# ====================== 07. Export Current Notebook 01 =======================

# ---------------------- 1. Check Complete Stage Outputs ----------------------
required_outputs = (
    "01_source_inventory.csv",
    "01_split_assignments.csv",
    "01_record_features.csv",
    "01_sequence_windows.npz",
    "01_sequence_normalisation.csv",
    "01_protocol.csv",
    "01_source_distribution.png",
    "01_source_acquisition_preflight.csv",
    "01_source_acquisition_task_summary.csv",
)
missing_outputs = [
    filename for filename in required_outputs
    if not (output_dir / filename).is_file()
]
if missing_outputs:
    raise FileNotFoundError(
        "Notebook 01 stage outputs are incomplete: "
        f"{missing_outputs}. Execute the preceding cells first."
    )

# ----------------------- 2. Git Size Policy Inspection -----------------------
for filename in required_outputs:
    n_bytes = (output_dir / filename).stat().st_size
    if n_bytes > 50_000_000:
        print(f"Keep outside Git: {filename} ({n_bytes:,} bytes)")

# -------------------------- 3. Export Stage-01 ZIP ---------------------------
download_outputs_from_colab("eye_movement_modelling/output", "01")
